# 03 — Projections

A **projection** simplifies a graph into a form useful for downstream
consumption: visualization, analysis, export. The library offers two
modes:

- **Graph-to-Graph (CONSTRUCT)**: stays in RDF. Result storable as a
  named graph in the holarchy.
- **Graph-to-Structure (Pythonic)**: exits RDF into dicts and LPG
  (labeled property graph) shapes. Useful for the "last mile" into
  visualization libraries.

This notebook covers both modes plus composable
`ProjectionPipeline`s.

For **RDF-declared, persistent projection pipelines** registered in
the holarchy registry, see `10_projection_plugins`.


## Setup


In [ ]:
from holonic import HolonicDataset
from holonic.projections import (
    CONSTRUCT_STRIP_TYPES,
    ProjectionPipeline,
    build_construct,
    project_to_lpg,
    strip_blank_nodes,
    localize_predicates,
)

ds = HolonicDataset()
ds.add_holon("urn:holon:x", "X")
ds.add_interior("urn:holon:x", '''
    @prefix ex: <urn:ex:> .
    <urn:item:1> a ex:Widget ;
        ex:label "Alpha" ;
        ex:relatedTo <urn:item:2> .
    <urn:item:2> a ex:Widget ;
        ex:label "Beta" .
    _:blank1 a ex:Hidden .
''')


## Mode 1 — Graph-to-Graph (CONSTRUCT)

Strip types but retain other structure. Stays in RDF; result can be
written back as a projection layer in the holarchy.


In [ ]:
query = build_construct(CONSTRUCT_STRIP_TYPES, graph_iri="urn:holon:x/interior")
result_graph = ds.backend.construct(query)
print(f"original interior: {len(ds.backend.get_graph('urn:holon:x/interior'))} triples")
print(f"typeless view:     {len(result_graph)} triples")


## Mode 2 — Graph-to-Structure (LPG projection)

`project_to_lpg` collapses RDF into a labeled-property-graph shape
with independent controls for types, literals, blank nodes, and
rdf:Lists.


In [ ]:
interior = ds.backend.get_graph("urn:holon:x/interior")
lpg = project_to_lpg(
    interior,
    collapse_types=True,      # rdf:type → node.types list
    collapse_literals=True,   # literals → node.attributes dict
    resolve_blanks=True,      # blank nodes → nested dicts
    resolve_lists=True,       # rdf:first/rest → Python lists
)
lpg.to_dict()


## Composable pipelines

`ProjectionPipeline` chains named steps (CONSTRUCTs + Python
transforms). The pipeline is ephemeral — constructed per-call and
applied once. For **persistent** pipelines registered in the
holarchy, see `10_projection_plugins`.


In [ ]:
pipeline = (
    ProjectionPipeline("viz-prep")
    .add_transform("strip-blanks", strip_blank_nodes)
    .add_transform("localize", localize_predicates)
)

result = pipeline.apply(interior)
for s, p, o in result:
    print(f"{s}  {p}  {o}")


## Per-holon convenience: `project_holon`

Merges all interior graphs of a holon, applies LPG projection,
optionally stores the result as a new projection layer.


In [ ]:
lpg = ds.project_holon("urn:holon:x",
                       store_as="urn:holon:x/projection/viz",
                       collapse_types=True,
                       collapse_literals=True)
print(f"nodes: {len(lpg.nodes)}  edges: {len(lpg.edges)}")


## Holarchy topology projection

`project_holarchy()` produces an LPG of the holarchy itself: holons
as nodes, portals and `cga:memberOf` as edges. Useful for
visualization of the holarchy structure (see `12_holarchy_viz`).


In [ ]:
ds.add_holon("urn:holon:y", "Y", member_of="urn:holon:x")
ds.add_portal("urn:portal:x-to-y", "urn:holon:x", "urn:holon:y",
              "CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
              label="X → Y")

topo = ds.project_holarchy()
print(f"holarchy nodes: {len(topo.nodes)}")
print(f"holarchy edges: {len(topo.edges)}")


## Where to go next

- `12_holarchy_viz` — render the topology using viz builders
- `10_projection_plugins` — register pipelines as RDF-persistent specs
  with provenance tracking
